# iPhone prices analysis — article charts

Charts for **Are iPhones Really More Expensive? An Inflation-Adjusted Dashboard in Python**.

- Fits and trend lines use models released **2007–2025**.
- **2026** models are shown as highlighted points only.
- Prices are in **August 2026 dollars** (`price_usd_adjusted`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MultipleLocator
from scipy import stats

DATA_FILE = "iphone_prices_adjusted.csv"
SOURCE = "Data: MLJAR, US BLS CPI-U"

COLORS = {
    "Standard": "#0f766e",
    "Plus": "#0891b2",
    "Pro": "#2563eb",
    "Pro Max": "#7c3aed",
    "Budget": "#d97706",
    "Other": "#9ca3af",
    "2026": "#ea580c",
    "original": "#9ca3af",
    "text": "#12234a",
    "muted": "#6b7280",
    "connector": "#94a3b8",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#9ca3af",
    "axes.grid": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "font.size": 11,
})


def new_chart():
    """1600 x 900 px chart with room for title, subtitle and source."""
    fig, ax = plt.subplots(figsize=(10, 5.625))
    fig.subplots_adjust(left=0.08, right=0.96, top=0.82, bottom=0.14)
    return fig, ax


def usd_axis(ax, top):
    ax.set_ylim(0, top)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:,.0f}"))


def year_axis(ax, start=2006, end=2026.6):
    ax.set_xlim(start, end)
    ax.xaxis.set_major_locator(MultipleLocator(2))
    ax.set_xlabel("Release year")


def label(ax, text, xy, xytext, color=COLORS["text"]):
    """Text label connected to a point with a thin line (xytext in data coordinates)."""
    ax.annotate(text, xy, xytext=xytext, fontsize=9.5, color=color,
                arrowprops=dict(arrowstyle="-", color=COLORS["connector"], lw=0.9))


def plot_with_gaps(ax, x, y, **kwargs):
    """Line that breaks where release years are more than 1 year apart."""
    xs, ys = [], []
    for xi, yi in zip(x, y):
        if xs and xi - xs[-1] > 1:
            xs.append(np.nan)
            ys.append(np.nan)
        xs.append(xi)
        ys.append(yi)
    ax.plot(xs, ys, **kwargs)


def save_chart(fig, title, subtitle, filename):
    fig.text(0.08, 0.93, title, fontsize=16, fontweight="bold", color=COLORS["text"])
    fig.text(0.08, 0.875, subtitle, fontsize=10.5, color=COLORS["muted"])
    fig.text(0.08, 0.025, SOURCE, fontsize=9, color=COLORS["muted"])
    fig.savefig(filename, dpi=160)
    plt.show()

In [ ]:
df = pd.read_csv(DATA_FILE)


def model_line(model):
    if "SE" in model or model in ["iPhone 5c", "iPhone 16e", "iPhone 17e"]:
        return "Budget"
    if "Pro Max" in model or model == "iPhone XS Max":
        return "Pro Max"
    if "Pro" in model or model in ["iPhone X", "iPhone XS"]:
        return "Pro"
    if "Plus" in model:
        return "Plus"
    if "mini" in model or model in ["iPhone Air", "iPhone Duo"]:
        return "Other"
    return "Standard"


df["model_line"] = df["model"].map(model_line)
df["price_per_sq_inch"] = df["price_usd_adjusted"] / df["screen_area_sq_inches"]
df["price_per_sq_inch_total"] = df["price_usd_adjusted"] / df["total_screen_area_sq_inches"]

priced = df.dropna(subset=["price_usd"])
hist = priced[priced["release_year"] <= 2025]  # used for fits and trends
new = priced[priced["release_year"] == 2026]   # highlighted points only


def row(model):
    return df.loc[df["model"] == model].iloc[0]


print(f"{len(df)} models · {len(hist)} priced models 2007–2025 · {len(new)} models in 2026")
df.groupby("model_line")["model"].count()

In [ ]:
# Chart 1: standard iPhone, original vs inflation-adjusted launch price
std = df[(df["model_line"] == "Standard") & (df["release_year"] <= 2025)].sort_values("release_year")

last_price = std["price_usd"].iloc[-1]
first_year = std.loc[std["price_usd"] != last_price, "release_year"].max() + 1
last_year = std["release_year"].max()

fig, ax = new_chart()

ax.axvspan(first_year - 0.4, last_year + 0.4, color=COLORS["Standard"], alpha=0.07, lw=0)
ax.text((first_year + last_year) / 2, 60, f"Price frozen at ${last_price:,.0f}",
        ha="center", fontsize=10, color=COLORS["Standard"])

ax.plot(std["release_year"], std["price_usd"], "o-", color=COLORS["original"], lw=2, ms=5,
        label="Original price")
ax.plot(std["release_year"], std["price_usd_adjusted"], "o-", color=COLORS["Standard"], lw=2.6, ms=6,
        label="Inflation-adjusted price")

for model in ["iPhone (1st generation)", "iPhone 4", "iPhone 12", "iPhone 17"]:
    r = row(model)
    name = "Original iPhone" if model == "iPhone (1st generation)" else model
    ax.annotate(f"{name}\n${r['price_usd_adjusted']:,.0f}",
                (r["release_year"], r["price_usd_adjusted"]),
                xytext=(0, 10), textcoords="offset points", ha="center",
                fontsize=9.5, color=COLORS["text"])

usd_axis(ax, 1200)
year_axis(ax)
ax.legend(frameon=False, loc="lower left")

save_chart(
    fig,
    "Standard iPhone launch price, 2007–2025",
    "Original and inflation-adjusted prices · adjusted prices in August 2026 dollars",
    "iphone-prices-standard-original-vs-adjusted.png",
)

In [ ]:
# Chart 2: inflation-adjusted launch price by model line
fig, ax = new_chart()
fig.subplots_adjust(right=0.9)

for line in ["Pro Max", "Pro", "Plus", "Standard", "Budget"]:
    color = COLORS[line]
    h = hist[hist["model_line"] == line].sort_values("release_year")
    plot_with_gaps(ax, h["release_year"], h["price_usd_adjusted"], color=color, lw=2.4)
    ax.plot(h["release_year"], h["price_usd_adjusted"], "o", color=color, ms=6)
    end = h.iloc[-1]

    # 2026 model of this line: hollow marker joined with a dashed segment
    n = new[new["model_line"] == line]
    if len(n):
        n = n.iloc[0]
        ax.plot([end["release_year"], n["release_year"]],
                [end["price_usd_adjusted"], n["price_usd_adjusted"]],
                "--", color=color, lw=1.6)
        ax.plot(n["release_year"], n["price_usd_adjusted"], "o",
                mfc="white", mec=color, mew=2, ms=8)
        end = n

    ax.annotate(line, (end["release_year"], end["price_usd_adjusted"]),
                xytext=(10, -4), textcoords="offset points",
                fontsize=10.5, fontweight="bold", color=color, annotation_clip=False)

# iPhone Duo is far above the scale: arrow at the top edge instead of a point
Y_TOP = 1600
duo = row("iPhone Duo")
ax.annotate("", xy=(duo["release_year"], Y_TOP), xytext=(duo["release_year"], Y_TOP - 130),
            arrowprops=dict(arrowstyle="-|>", color=COLORS["2026"], lw=2, mutation_scale=16),
            annotation_clip=False)
ax.text(duo["release_year"] - 0.25, Y_TOP - 75, f"iPhone Duo: ${duo['price_usd_adjusted']:,.0f}",
        ha="right", va="center", fontsize=10.5, fontweight="bold", color=COLORS["2026"])

ax.legend(handles=[Line2D([], [], marker="o", ls="", mfc="white", mec=COLORS["muted"],
                          mew=2, ms=8, label="2026 models")],
          frameon=False, loc="lower left")
usd_axis(ax, Y_TOP)
year_axis(ax)

save_chart(
    fig,
    "iPhone launch price by model line, 2007–2026",
    "Inflation-adjusted prices in August 2026 dollars · hollow markers are 2026 models",
    "iphone-prices-by-model-line.png",
)

In [ ]:
# Chart 3: screen area vs inflation-adjusted price
# Linear fit uses 2007–2025 models; 2026 models are shown as highlighted points only.
x = hist["screen_area_sq_inches"]
y = hist["price_usd_adjusted"]
n = len(hist)

fit = stats.linregress(x, y)
xs = np.linspace(x.min(), x.max(), 100)
y_fit = fit.intercept + fit.slope * xs

# 95% confidence interval for the mean linear trend
resid_sd = np.sqrt(np.sum((y - (fit.intercept + fit.slope * x)) ** 2) / (n - 2))
se_mean = resid_sd * np.sqrt(1 / n + (xs - x.mean()) ** 2 / np.sum((x - x.mean()) ** 2))
t = stats.t.ppf(0.975, n - 2)

fig, ax = new_chart()
ax.fill_between(xs, y_fit - t * se_mean, y_fit + t * se_mean,
                color=COLORS["connector"], alpha=0.2, lw=0, label="95% confidence interval")
ax.plot(xs, y_fit, color=COLORS["text"], lw=2, label="Linear fit (2007–2025)")

for line in ["Standard", "Plus", "Pro", "Pro Max", "Budget", "Other"]:
    g = hist[hist["model_line"] == line]
    ax.scatter(g["screen_area_sq_inches"], g["price_usd_adjusted"], s=50,
               color=COLORS[line], edgecolor="white", lw=0.8, zorder=3, label=line)

ax.scatter(new["screen_area_sq_inches"], new["price_usd_adjusted"], s=60, color=COLORS["2026"],
           edgecolor="white", lw=0.8, zorder=4, label="2026 models")

label_positions = {
    "iPhone X": (8.2, 1440),
    "iPhone XS Max": (16.8, 1680),
    "iPhone SE (3rd generation)": (5.2, 300),
    "iPhone 16e": (16.0, 470),
    "iPhone 17": (17.2, 700),
    "iPhone 17 Pro Max": (19.6, 1150),
    "iPhone Duo": (20.6, 2050),
}
for model, pos in label_positions.items():
    r = row(model)
    color = COLORS["2026"] if r["release_year"] == 2026 else COLORS["text"]
    label(ax, f"{model}: ${r['price_usd_adjusted']:,.0f}",
          (r["screen_area_sq_inches"], r["price_usd_adjusted"]), pos, color)

ax.set_xlim(4, 29)
ax.set_xlabel("Screen area (square inches)")
usd_axis(ax, 2200)
ax.legend(frameon=False, loc="upper left", ncol=2, fontsize=9.5)

save_chart(
    fig,
    "iPhone screen area and launch price",
    f"Inflation-adjusted prices in August 2026 dollars · linear fit on 2007–2025 models: "
    f"+${fit.slope:.0f} per square inch, R² {fit.rvalue ** 2:.2f}",
    "iphone-prices-screen-area.png",
)

In [ ]:
# Chart 4: inflation-adjusted price per square inch of screen
b = np.polyfit(hist["release_year"], np.log(hist["price_per_sq_inch"]), 1)
yearly_change = np.exp(b[0]) - 1


def trend(year):
    return np.exp(np.polyval(b, year))


fig, ax = new_chart()

years = np.linspace(2007, 2025, 100)
ax.plot(years, trend(years), color=COLORS["Standard"], lw=2.6, zorder=2,
        label=f"Trend 2007–2025: {yearly_change:+.1%} per year")
ax.plot([2025, 2026], trend(np.array([2025, 2026])), color=COLORS["Standard"], lw=2, ls="--", zorder=2)

ax.scatter(hist["release_year"], hist["price_per_sq_inch"], s=38, color=COLORS["Pro"],
           edgecolor="white", lw=0.8, zorder=3, label="2007–2025 models")

phones_2026 = new[new["model"] != "iPhone Duo"]
ax.scatter(phones_2026["release_year"], phones_2026["price_per_sq_inch"], s=52, color=COLORS["2026"],
           edgecolor="white", lw=0.8, zorder=4, label="2026 models")

# iPhone Duo: inner screen vs inner + outer screen
duo = row("iPhone Duo")
duo_x = duo["release_year"] + 0.3  # small offset so the Duo does not cover the 18 Pro Max
ax.plot([duo_x, duo_x], [duo["price_per_sq_inch"], duo["price_per_sq_inch_total"]],
        ls=":", color=COLORS["2026"], lw=1.3, zorder=4)
ax.scatter(duo_x, duo["price_per_sq_inch"], marker="D", s=56, color=COLORS["2026"],
           edgecolor="white", zorder=5, label="iPhone Duo, inner screen")
ax.scatter(duo_x, duo["price_per_sq_inch_total"], marker="D", s=56, facecolor="white",
           edgecolor=COLORS["2026"], lw=1.8, zorder=5, label="iPhone Duo, inner + outer screen")

for model, pos, color in [
    ("iPhone 4", (2011.3, 180), COLORS["text"]),
    ("iPhone 17", (2016.5, 32), COLORS["text"]),
    ("iPhone 18 Pro", (2019.8, 140), COLORS["2026"]),
    ("iPhone 17e", (2021.8, 8), COLORS["2026"]),
]:
    r = row(model)
    label(ax, f"{model}: ${r['price_per_sq_inch']:.0f}/in²",
          (r["release_year"], r["price_per_sq_inch"]), pos, color)

label(ax, f"iPhone Duo, inner screen: ${duo['price_per_sq_inch']:.0f}/in²",
      (duo_x, duo["price_per_sq_inch"]), (2018.4, 122), COLORS["2026"])
label(ax, f"iPhone Duo, inner + outer: ${duo['price_per_sq_inch_total']:.0f}/in²",
      (duo_x, duo["price_per_sq_inch_total"]), (2018.6, 20), COLORS["2026"])

usd_axis(ax, 190)
year_axis(ax, 2006, 2027)
ax.legend(frameon=False, loc="lower left", fontsize=9.5)

save_chart(
    fig,
    "iPhone price per square inch of screen, 2007–2026",
    "Inflation-adjusted prices in August 2026 dollars · trend fitted on 2007–2025 models",
    "iphone-prices-per-square-inch.png",
)